In [1]:
import numpy as  np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
df = pd.read_csv("ipl_clean_data.csv")

C:\Users\ASUS\AppData\Local\Temp\ipykernel_11244\542382566.py:1: DtypeWarning: Columns (1) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("ipl_clean_data.csv")


In [3]:
df.head(10)

,match_id,season,date,venue,city,innings,batting_team,bowling_team,over,bowler_wicket,...,runs_total,runs_extras,valid_ball,wicket_kind,player_out,fielders,toss_winner,toss_decision,match_won_by,runs_target
0,335982,2007/08,2008-04-18,M Chinnaswamy Stadium,Bangalore,1,Kolkata Knight Riders,Royal Challengers Bangalore,0,0,...,1,1,1,NaN,NaN,NaN,Royal Challengers Bangalore,field,Kolkata Knight Riders,NaN
1,335982,2007/08,2008-04-18,M Chinnaswamy Stadium,Bangalore,1,Kolkata Knight Riders,Royal Challengers Bangalore,0,0,...,0,0,1,NaN,NaN,NaN,Royal Challengers Bangalore,field,Kolkata Knight Riders,NaN
2,335982,2007/08,2008-04-18,M Chinnaswamy Stadium,Bangalore,1,Kolkata Knight Riders,Royal Challengers Bangalore,0,0,...,1,1,0,NaN,NaN,NaN,Royal Challengers Bangalore,field,Kolkata Knight Riders,NaN
3,335982,2007/08,2008-04-18,M Chinnaswamy Stadium,Bangalore,1,Kolkata Knight Riders,Royal Challengers Bangalore,0,0,...,0,0,1,NaN,NaN,NaN,Royal Challengers Bangalore,field,Kolkata Knight Riders,NaN
4,335982,2007/08,2008-04-18,M Chinnaswamy Stadium,Bangalore,1,Kolkata Knight Riders,Royal Challengers Bangalore,0,0,...,0,0,1,NaN,NaN,NaN,Royal Challengers Bangalore,field,Kolkata Knight Riders,NaN
5,335982,2007/08,2008-04-18,M Chinnaswamy Stadium,Bangalore,1,Kolkata Knight Riders,Royal Challengers Bangalore,0,0,...,0,0,1,NaN,NaN,NaN,Royal Challengers Bangalore,field,Kolkata Knight Riders,NaN
6,335982,2007/08,2008-04-18,M Chinnaswamy Stadium,Bangalore,1,Kolkata Knight Riders,Royal Challengers Bangalore,0,0,...,1,1,1,NaN,NaN,NaN,Royal Challengers Bangalore,field,Kolkata Knight Riders,NaN
7,335982,2007/08,2008-04-18,M Chinnaswamy Stadium,Bangalore,1,Kolkata Knight Riders,Royal Challengers Bangalore,1,0,...,0,0,1,NaN,NaN,NaN,Royal Challengers Bangalore,field,Kolkata Knight Riders,NaN
8,335982,2007/08,2008-04-18,M Chinnaswamy Stadium,Bangalore,1,Kolkata Knight Riders,Royal Challengers Bangalore,1,0,...,4,0,1,NaN,NaN,NaN,Royal Challengers Bangalore,field,Kolkata Knight Riders,NaN
9,335982,2007/08,2008-04-18,M Chinnaswamy Stadium,Bangalore,1,Kolkata Knight Riders,Royal Challengers Bangalore,1,0,...,4,0,1,NaN,NaN,NaN,Royal Challengers Bangalore,field,Kolkata Knight Riders,NaN


In [4]:
df["wicket_kind"].value_counts()

wicket_kind
caught                   8665
bowled                   2345
run out                  1153
lbw                       853
caught and bowled         388
stumped                   376
hit wicket                 18
retired hurt               17
retired out                 5
obstructing the field       3
Name: count, dtype: int64

In [5]:
bowler_wickets = df[df["wicket_kind"].isin(['caught', 'bowled', 'lbw', 'caught and bowled', 'stumped', 'hit wicket'])]
top_wicket_taker = bowler_wickets.groupby("bowler")["wicket_kind"].count().sort_values(ascending = False).reset_index()
top_wicket_taker.to_csv("top_wicket_taker.csv",index=False)

In [6]:
bowler_balls = df.groupby("bowler").size()

In [7]:
bowler_balls.sort_values(ascending=False).head(10)

bowler
R Ashwin           4868
SP Narine          4421
B Kumar            4378
RA Jadeja          4127
YS Chahal          3905
PP Chawla          3895
Harbhajan Singh    3496
JJ Bumrah          3474
A Mishra           3444
AR Patel           3371
dtype: int64

In [8]:
# 1. Groupby se calculations karo (Isme bowler index ban jayega)
bowler_runs = df.groupby("bowler")["runs_total"].sum()
bowler_balls = df.groupby("bowler")["valid_ball"].sum()
economy = (bowler_runs / bowler_balls) * 6

# 2. DataFrame banao (index apne aap 'bowler' rahega)
economy_df = pd.DataFrame({
    "runs conceded": bowler_runs,
    "balls": bowler_balls,
    "economy": economy
})

# 3. Filter karo (Balls >= 1000)
economy_df = economy_df[economy_df["balls"] >= 1000]

# 4. Sort karo, reset_index karo (taaki bowler column ban jaye), aur round off
economy_df = economy_df.sort_values("economy", ascending=True)
economy_df = economy_df.reset_index()
economy_df["economy"] = economy_df["economy"].round(2)

# 5. Save to CSV (Saare columns save honge ab)
economy_df.to_csv("best_economy.csv", index=False)

# Check output
economy_df.head(10)

,bowler,runs conceded,balls,economy
0,M Muralitharan,1765,1528,6.93
1,SP Narine,5029,4351,6.93
2,DW Steyn,2583,2182,7.10
3,Harbhajan Singh,4101,3416,7.20
4,Rashid Khan,3863,3202,7.24
5,R Ashwin,5721,4710,7.29
6,SK Warne,1465,1194,7.36
7,SL Malinga,3486,2827,7.40
8,M Kartik,1418,1149,7.40
9,JJ Bumrah,4162,3359,7.43


In [9]:
# Filtering for Powerplay wickets (Over 0 to 5)
powerplay_wickets = df[(df['over'] < 6) & (df['bowler_wicket'] == True)]
pp_specialists = powerplay_wickets.groupby('bowler')['bowler_wicket'].count().sort_values(ascending=False)
pp_specialists = pp_specialists.reset_index()
pp_specialists.to_csv("PP_wicket_takers.csv",index=False)
pp_specialists.head(10)

,bowler,bowler_wicket
0,B Kumar,80
1,TA Boult,72
2,DL Chahar,66
3,Sandeep Sharma,62
4,UT Yadav,58
5,I Sharma,57
6,Z Khan,52
7,Mohammed Shami,51
8,R Ashwin,50
9,Mohammed Siraj,45


In [10]:
death_wicket = df[(df['over'] > 16) & (df['bowler_wicket'] == True)]
death_wicket_taker = death_wicket.groupby('bowler')['bowler_wicket'].count().sort_values(ascending = False)
death_wicket_taker = death_wicket_taker.reset_index()
death_wicket_taker.to_csv("death_specialist.csv",index=False)
death_wicket_taker.head(10)

,bowler,bowler_wicket
0,DJ Bravo,90
1,B Kumar,76
2,SL Malinga,76
3,HV Patel,64
4,JJ Bumrah,63
5,MM Sharma,54
6,Mohammed Shami,48
7,TA Boult,47
8,K Rabada,46
9,SP Narine,45


In [11]:
middle_wickets = df[(df['over'] <= 16) & (df['over'] > 6) & (df['bowler_wicket'] == True)]
middle_wicket_taker = middle_wickets.groupby("bowler")["bowler_wicket"].count().sort_values(ascending=False)
middle_wicket_taker = middle_wicket_taker.reset_index()
middle_wicket_taker.to_csv("middle_over_specialist.csv",index=False)
middle_wicket_taker.head(10)

,bowler,bowler_wicket
0,YS Chahal,162
1,PP Chawla,144
2,A Mishra,132
3,RA Jadeja,132
4,R Ashwin,122
5,Rashid Khan,118
6,SP Narine,117
7,Harbhajan Singh,96
8,AR Patel,86
9,Kuldeep Yadav,83


In [12]:
dot_balls = df[df['runs_total']==0]

dot_counts = dot_balls.groupby("bowler").size()

total_balls = df.groupby("bowler").size()

dot_percentage = (dot_counts/total_balls)*100

dot_df = pd.DataFrame({
    "dot balls" : dot_counts,
    "balls":total_balls,
    "dot percentage":dot_percentage
}
                     )

dot_df = dot_df[dot_df["balls"]>1000]
dot_df=dot_df.reset_index()
dot_df.to_csv("most_dot_balls.csv",index=False)

dot_df.sort_values("dot balls", ascending= False).head(10)

,bowler,dot balls,balls,dot percentage
8,B Kumar,1748.0,4378,39.926907
69,SP Narine,1653.0,4421,37.389731
53,R Ashwin,1606.0,4868,32.990961
26,JJ Bumrah,1353.0,3474,38.946459
51,PP Chawla,1325.0,3895,34.017972
56,RA Jadeja,1292.0,4127,31.306033
81,YS Chahal,1266.0,3905,32.419974
18,Harbhajan Singh,1263.0,3496,36.127002
77,UT Yadav,1186.0,3190,37.178683
0,A Mishra,1185.0,3444,34.407666


In [18]:
# 1. Match aur Bowler ke hisaab se runs calculate karo
bowler_match_runs = df.groupby(['match_id', 'bowler', 'bowling_team', 'batting_team'])['runs_total'].sum().reset_index()

# 2. Highest se Lowest sort karo
highest_runs_conceded = bowler_match_runs.sort_values(by='runs_total', ascending=False)
highest_run_conceded = highest_runs_conceded.reset_index()
highest_run_conceded.to_csv("most_expensive_bowler.csv",index=False)

print("--- Highest Runs Conceded by a Bowler (Single Match) ---")
highest_runs_conceded

--- Highest Runs Conceded by a Bowler (Single Match) ---


,match_id,bowler,bowling_team,batting_team,runs_total
12993,1473439,JC Archer,Rajasthan Royals,Sunrisers Hyderabad,80
13293,1473464,Mohammed Shami,Sunrisers Hyderabad,Punjab Kings,75
13786,1473507,W O'Rourke,Lucknow Super Giants,Royal Challengers Bengaluru,74
12594,1426278,MM Sharma,Gujarat Titans,Delhi Capitals,73
8169,1136611,Basil Thampi,Sunrisers Hyderabad,Royal Challengers Bangalore,70
...,...,...,...,...,...
10448,1304054,AD Russell,Kolkata Knight Riders,Punjab Kings,0
2097,419163,SK Raina,Chennai Super Kings,Deccan Chargers,0
6082,829813,Z Khan,Delhi Daredevils,Royal Challengers Bangalore,0
7776,1136578,N Rana,Kolkata Knight Riders,Kings XI Punjab,0


In [19]:
# 1. Over-wise runs calculate karo
# Hum match_id, innings aur over (number) ko group karenge
expensive_overs = df.groupby(['match_id', 'over', 'bowler', 'bowling_team', 'batting_team'])['runs_total'].sum().reset_index()

# 2. Sort karke top 10 expensive overs nikaalo
most_expensive_overs = expensive_overs.sort_values(by='runs_total', ascending=False)
most_expensive_overs = most_expensive_overs.reset_index()
most_expensive_overs.to_csv("expensive_bowler.csv",index=False)

print("--- Most Expensive Overs in IPL History ---")
most_expensive_overs

--- Most Expensive Overs in IPL History ---


,index,match_id,over,bowler,bowling_team,batting_team,runs_total
0,32158,1254076,19,HV Patel,Royal Challengers Bangalore,Chennai Super Kings,37
1,8581,501247,2,P Parameswaran,Kochi Tuskers Kerala,Royal Challengers Bangalore,37
2,34286,1304060,15,DR Sams,Mumbai Indians,Kolkata Knight Riders,35
3,5702,419139,12,RS Bopara,Kings XI Punjab,Kolkata Knight Riders,33
4,17587,734047,5,P Awana,Kings XI Punjab,Chennai Super Kings,33
...,...,...,...,...,...,...,...
45022,38093,1359512,2,A Badoni,Lucknow Super Giants,Punjab Kings,0
45023,20164,980917,3,DS Kulkarni,Gujarat Lions,Mumbai Indians,0
45024,29157,1216494,11,Washington Sundar,Royal Challengers Bangalore,Kolkata Knight Riders,0
45025,382,335991,19,IK Pathan,Kings XI Punjab,Mumbai Indians,0
